In [1]:
import os
os.chdir(r"D:\product-analytics-ecommerce")
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

cats = pd.read_csv("data/raw/cookie_cats.csv")
cats.groupby("version").agg(
    players=("userid", "count"),
    day1_retention=("retention_1", "mean"),
    day7_retention=("retention_7", "mean"),
    avg_rounds=("sum_gamerounds", "mean"),
)

,players,day1_retention,day7_retention,avg_rounds
version,,,,
gate_30,44700,0.448188,0.190201,52.456264
gate_40,45489,0.442283,0.182000,51.298776


In [2]:
a = cats[cats.version == "gate_30"].retention_7
b = cats[cats.version == "gate_40"].retention_7

successes = [a.sum(), b.sum()]
n = [len(a), len(b)]

z, p = proportions_ztest(successes, n)
diff = b.mean() - a.mean()

print(f"gate_30 day-7 retention: {a.mean():.2%}")
print(f"gate_40 day-7 retention: {b.mean():.2%}")
print(f"Difference: {diff:.2%}")
print(f"Relative change: {diff / a.mean():.1%}")
print(f"p-value: {p:.4f}")

gate_30 day-7 retention: 19.02%
gate_40 day-7 retention: 18.20%
Difference: -0.82%
Relative change: -4.3%
p-value: 0.0016


In [3]:
se = np.sqrt(a.mean()*(1-a.mean())/len(a) + b.mean()*(1-b.mean())/len(b))
low, high = diff - 1.96*se, diff + 1.96*se
print(f"95% CI for difference: [{low:.2%}, {high:.2%}]")

95% CI for difference: [-1.33%, -0.31%]


In [4]:
rng = np.random.default_rng(42)
a_vals, b_vals = a.values.astype(int), b.values.astype(int)
diffs = [
    rng.choice(b_vals, len(b_vals)).mean() - rng.choice(a_vals, len(a_vals)).mean()
    for _ in range(5000)
]
print(f"Bootstrap 95% interval: [{np.percentile(diffs, 2.5):.2%}, {np.percentile(diffs, 97.5):.2%}]")

Bootstrap 95% interval: [-1.33%, -0.32%]
